# Seismic Facies Classification — Bryne Formation

End-to-end pipeline:
1. Ingest SEG-Y attribute volumes → NetCDF
2. Merge attributes + horizons into a single volume
3. Horizon sculpt to isolate the Bryne Formation
4. Train Self-Organizing Map (SOM) and export classification cube

## Imports & Constants

In [ ]:
import pathlib

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
import xarray as xr

from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
from minisom import MiniSom

In [ ]:
# ── Well location ──────────────────────────────────────────────────────────────
WELL_ILINE  = 4272
WELL_XLINE  = 10486
WELL_XY     = [617263.72, 6392408.28]

# ── Spatial crop around well (±lines) ─────────────────────────────────────────
CROP_WINDOW = 200

# ── Bryne Formation horizon sculpt window (samples / ms TWT) ──────────────────
MASK_ABOVE  = 20   # samples above Bryne horizon
MASK_BELOW  = 150  # samples below Bryne horizon

# ── SOM hyperparameters ────────────────────────────────────────────────────────
SOM_X           = 5
SOM_Y           = 5
SOM_ALPHA       = 0.5
SOM_SIGMA0      = 1.5
SOM_ITERATIONS  = 5000
RANDOM_SEED     = 123

# ── Paths ──────────────────────────────────────────────────────────────────────
ATTR_DIR    = pathlib.Path('Attributes')
VOL_DIR     = pathlib.Path('VOLUMES')
MERGED_PATH = ATTR_DIR / 'Attributes_Horizons.seisnc'
PETREL_VOL  = VOL_DIR  / 'Volume_Attributes.seisnc'

np.random.seed(RANDOM_SEED)

---
## Step 1 — Ingest SEG-Y Attribute Volumes

Reads each Petrel-exported SEG-Y and saves it as a NetCDF file.  
Byte locations: iline=5, xline=21, cdp_x=73, cdp_y=77.

In [ ]:
# Map: SEG-Y filename → output NetCDF filename
SEGY_TO_NC = {
    'amplitude.segy'         : 'Amplitudes.nc',
    '3D_edge_enh.segy'       : 'Edge_3D_enhance.nc',
    'AppPol_w33.segy'        : 'Apparent_polarity_w33.nc',
    'DomFreq_w33.segy'       : 'Dominant_frequency_w33.nc',
    'Env_w33.segy'           : 'Envelope_w33.nc',
    'GLCM_I.segy'            : 'GLCM_I.nc',
    'InstPhase_w33.segy'     : 'Instantaneous_phase_w33.nc',
    'RAI_lc10.segy'          : 'RAI_lc10.nc',
    'Struct_smooth_1pt2.segy': 'Structural_smooth_1pt2.nc',
    'Sweet_w33.segy'         : 'Sweetness_w33.nc',
    'Variance.segy'          : 'Variance.nc',
    '3D_curve.segy'          : 'Curvature3D.nc',
    'Chaos.segy'             : 'Chaos.nc',
}

for segy_name, nc_name in SEGY_TO_NC.items():
    segy_path = ATTR_DIR / segy_name
    nc_path   = ATTR_DIR / nc_name

    if nc_path.exists():
        print(f'  skip  {nc_name}  (already converted)')
        continue

    print(f'  loading  {segy_name} ... ', end='')
    ds = xr.open_dataset(
        segy_path,
        dim_byte_fields={'iline': 5, 'xline': 21},
        extra_byte_fields={'cdp_x': 73, 'cdp_y': 77},
    )
    ds = ds.set_coords(('cdp_x', 'cdp_y'))
    ds.to_netcdf(nc_path)
    ds.close()
    print(f'saved → {nc_name}')

print('\nStep 1 complete.')

---
## Step 2 — Merge Attributes + Horizons

Combines all per-attribute NetCDF files into one xarray Dataset and attaches
the three geological horizon surfaces (Tau, Sandnes, Bryne TWT) as data variables.

In [ ]:
# Map: NetCDF filename → variable name in the merged volume
NC_TO_VAR = {
    'Amplitudes.nc'              : 'data',       # base amplitude — kept as 'data'
    'Apparent_polarity_w33.nc'   : 'apppol',
    'Chaos.nc'                   : 'chaos',
    'Curvature3D.nc'             : 'curve',
    'Dominant_frequency_w33.nc'  : 'dfreq',
    'Edge_3D_enhance.nc'         : 'edge',
    'Envelope_w33.nc'            : 'env',
    'GLCM_I.nc'                  : 'glcmi',
    'Instantaneous_phase_w33.nc' : 'iphase',
    'RAI_lc10.nc'                : 'rai',
    'Structural_smooth_1pt2.nc'  : 'ssmooth',
    'Sweetness_w33.nc'           : 'sweet',
    'Variance.nc'                : 'var',
}

# Load amplitude as the base volume
with xr.open_dataset(ATTR_DIR / 'Amplitudes.nc') as ds:
    volume = ds.load()

# Merge remaining attributes
for nc_name, var_name in NC_TO_VAR.items():
    if var_name == 'data':          # amplitude already loaded as base
        continue
    with xr.open_dataset(ATTR_DIR / nc_name) as ds:
        volume[var_name] = ds.data.load()
    print(f'  merged  {var_name}')

# Attach horizon surfaces from the Petrel volume
with xr.open_dataset(PETREL_VOL) as petrel:
    volume['Tau_TWT']    = petrel.Tau_TWT.load()
    volume['Sandnes_TWT']= petrel.Sandnes_TWT.load()
    volume['Bryne_TWT']  = petrel.Bryne_TWT.load()

print(f'\nMerged volume: {dict(volume.dims)}')
print(f'Variables: {list(volume.data_vars)}')

volume.to_netcdf(MERGED_PATH)
print(f'\nSaved → {MERGED_PATH}')

---
## Step 3 — Horizon Sculpt: Isolate Bryne Formation

Crops the merged volume spatially around the well, then masks to the depth window
defined by the Bryne horizon (MASK_ABOVE samples above → MASK_BELOW samples below).

In [ ]:
volume = xr.open_dataset(MERGED_PATH)

# Spatial crop around the well
cube = volume.sel(
    iline=slice(WELL_ILINE - CROP_WINDOW, WELL_ILINE + CROP_WINDOW),
    xline=slice(WELL_XLINE - CROP_WINDOW, WELL_XLINE + CROP_WINDOW),
)

print(f'Cropped cube: {dict(cube.dims)}')

In [ ]:
# Boolean mask: samples within [Bryne - MASK_ABOVE, Bryne + MASK_BELOW]
# NOTE: samples axis is positive-down (increasing TWT), opposite to Petrel depth convention
bryne_mask = (
    (cube.samples > cube.Bryne_TWT - MASK_ABOVE) &
    (cube.samples < cube.Bryne_TWT + MASK_BELOW)
)
masked_data = cube.where(bryne_mask)

print(f'Mask window: Bryne −{MASK_ABOVE} to +{MASK_BELOW} samples')

### Visualise Masked Attributes

In [ ]:
def plot_section(data_array, xline, samples_slice, cmap, cube, title=''):
    """Plot an inline/sample section with the three horizon overlays."""
    opt = dict(
        x='iline', y='samples',
        add_colorbar=True, interpolation='spline16',
        robust=True, yincrease=False, cmap=cmap,
    )
    f, ax = plt.subplots(figsize=(12, 8), constrained_layout=True)
    data_array.sel(xline=xline, samples=samples_slice).plot.imshow(ax=ax, **opt)

    horizons = [
        (cube.Tau_TWT,     'b',      'Tau'),
        (cube.Sandnes_TWT, 'orange', 'Sandnes'),
        (cube.Bryne_TWT,   'r',      'Bryne'),
    ]
    for horizon, color, label in horizons:
        h = horizon.sel(xline=xline)
        ax.plot(h.iline, h, color=color, label=label)

    ax.invert_xaxis()
    ax.legend(loc='upper right')
    if title:
        ax.set_title(title)
    return ax

In [ ]:
samples_window = slice(2000, 2500)

plot_section(masked_data.data,    WELL_XLINE, samples_window, 'seismic_r', cube, 'Amplitude')
plot_section(masked_data.apppol,  WELL_XLINE, samples_window, 'RdBu_r',    cube, 'Apparent Polarity')
plot_section(masked_data.chaos,   WELL_XLINE, samples_window, 'hot_r',     cube, 'Chaos')
plot_section(masked_data.curve,   WELL_XLINE, samples_window, 'coolwarm',  cube, 'Curvature 3D')
plot_section(masked_data.dfreq,   WELL_XLINE, samples_window, 'rainbow',   cube, 'Dominant Frequency')
plot_section(masked_data.edge,    WELL_XLINE, samples_window, 'gray_r',    cube, 'Edge Enhancement 3D')
plot_section(masked_data.env,     WELL_XLINE, samples_window, 'gist_ncar', cube, 'Envelope')
plot_section(masked_data.glcmi,   WELL_XLINE, samples_window, 'viridis',   cube, 'GLCM Intensity')
plot_section(masked_data.iphase,  WELL_XLINE, samples_window, 'hsv',       cube, 'Instantaneous Phase')
plot_section(masked_data.rai,     WELL_XLINE, samples_window, 'seismic_r', cube, 'RAI')
plot_section(masked_data.ssmooth, WELL_XLINE, samples_window, 'seismic_r', cube, 'Structural Smooth')
plot_section(masked_data.sweet,   WELL_XLINE, samples_window, 'gist_ncar', cube, 'Sweetness')
plot_section(masked_data.var,     WELL_XLINE, samples_window, 'hot_r',     cube, 'Variance')

plt.show()

---
## Step 4 — Feature Extraction & PCA-guided Selection

Builds a feature matrix from **all 13 seismic attributes**, runs PCA to reveal
correlation structure and attribute importance, then selects a subset of features
for SOM training based on the PCA loadings.

In [ ]:
# All attribute names from NC_TO_VAR (13 attributes, excluding horizon surfaces)
ALL_ATTRS = list(NC_TO_VAR.values())

# Build full feature matrix — non-NaN voxels only
valid_mask = masked_data[ALL_ATTRS[0]].notnull().values

X_all = np.stack(
    [masked_data[col].values[valid_mask] for col in ALL_ATTRS],
    axis=1
)

print(f'Feature matrix: {X_all.shape}  ({X_all.shape[0]:,} voxels × {X_all.shape[1]} attributes)')
print(f'Attributes: {ALL_ATTRS}')

In [ ]:
# Correlation heatmap — 10k random sample to keep it fast
sample_idx = np.random.choice(X_all.shape[0], size=10_000, replace=False)
X_sample   = X_all[sample_idx]

corr = pd.DataFrame(X_sample, columns=ALL_ATTRS).corr()

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Attribute correlation matrix (10k sample)')
plt.tight_layout()
plt.show()

In [ ]:
# PCA on all 13 attributes
pca_all = PCA(n_components=len(ALL_ATTRS))
pca_all.fit(X_all)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Explained variance
axes[0].bar(range(1, len(ALL_ATTRS) + 1), pca_all.explained_variance_ratio_ * 100,
            color='steelblue', alpha=0.8)
ax2 = axes[0].twinx()
ax2.plot(range(1, len(ALL_ATTRS) + 1),
         np.cumsum(pca_all.explained_variance_ratio_) * 100,
         'r-o', ms=5, label='Cumulative')
ax2.set_ylabel('Cumulative variance (%)', color='r')
ax2.tick_params(axis='y', colors='r')
ax2.set_ylim(0, 105)
axes[0].set_xlabel('Principal Component')
axes[0].set_ylabel('Variance explained (%)')
axes[0].set_title('PCA — Explained Variance')
axes[0].set_xticks(range(1, len(ALL_ATTRS) + 1))

# Loadings heatmap (first 6 PCs)
loadings = pd.DataFrame(
    pca_all.components_[:6].T,
    index=ALL_ATTRS,
    columns=[f'PC{i+1}' for i in range(6)],
)
sns.heatmap(loadings, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, ax=axes[1], linewidths=0.5, vmin=-1, vmax=1)
axes[1].set_title('PCA Loadings — first 6 PCs')

plt.tight_layout()
plt.show()

print('\nVariance explained per component:')
for i, var in enumerate(pca_all.explained_variance_ratio_):
    print(f'  PC{i+1:2d}: {var:5.1%}  (cumulative: {pca_all.explained_variance_ratio_[:i+1].sum():.1%})')

### Feature selection — PCA-ranked attributes

Each attribute is scored by its total contribution to explained variance:
`score_i = Σ_j (loading_ij² × variance_explained_j)`

The top 5 are automatically selected as `FEATURE_COLS`.
Override the list manually if domain knowledge suggests a different set.

In [ ]:
# Rank attributes by PCA importance:
# score_i = sum over PCs of (loading_ij^2 * explained_variance_ratio_j)
importance = (pca_all.components_ ** 2 * pca_all.explained_variance_ratio_[:, None]).sum(axis=0)
attr_ranking = sorted(zip(ALL_ATTRS, importance), key=lambda x: x[1], reverse=True)

print('Attribute ranking by PCA importance:')
for rank, (attr, score) in enumerate(attr_ranking, 1):
    print(f'  {rank:2d}.  {attr:<8s}  score={score:.4f}')

# Select top 5
N_FEATURES  = 5
FEATURE_COLS = [attr for attr, _ in attr_ranking[:N_FEATURES]]

# Subset the full feature matrix
feat_idx = [ALL_ATTRS.index(c) for c in FEATURE_COLS]
X_orig   = X_all[:, feat_idx]

print(f'\nSelected {N_FEATURES} features: {FEATURE_COLS}')
print(f'Feature matrix for SOM: {X_orig.shape}')


In [ ]:
# MinMax scale to [0, 1] for SOM training
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X_orig)

print(f'Scaled range: [{X_scaled.min():.2f}, {X_scaled.max():.2f}]')

---
## Step 5 — Train Self-Organizing Map

In [ ]:
som = MiniSom(
    SOM_X, SOM_Y,
    input_len=X_scaled.shape[1],
    sigma=SOM_SIGMA0,
    learning_rate=SOM_ALPHA,
    neighborhood_function='triangle',
    activation_distance='euclidean',
    topology='rectangular',
    sigma_decay_function='linear_decay_to_one',
    decay_function='linear_decay_to_zero',
    random_seed=RANDOM_SEED,
)

print(f'Training {SOM_X}×{SOM_Y} SOM  |  {SOM_ITERATIONS} iterations  |  {X_scaled.shape[0]:,} samples')
som.train_random(X_scaled, SOM_ITERATIONS, verbose=True)

In [ ]:
# U-matrix: inter-neuron distances — reveals cluster boundaries
plt.figure(figsize=(6, 5))
plt.pcolor(som.distance_map().T, cmap='gist_yarg')
plt.colorbar(label='Distance')
plt.axis('equal')
plt.title('SOM U-matrix')
plt.show()

---
## Step 6 — Assign Classification Labels

In [ ]:
# Vectorised BMU lookup via distance-from-weights
# Faster than a pure Python loop over 1.7M samples
dists      = som._distance_from_weights(X_scaled)       # shape: (n_samples, SOM_X*SOM_Y)
bmu_flat   = dists.argmin(axis=1)                       # flat index into SOM grid
bmu_coords = np.unravel_index(bmu_flat, (SOM_X, SOM_Y)) # (row_arr, col_arr)
labels     = np.ravel_multi_index(bmu_coords, (SOM_X, SOM_Y))  # 1D class label 0–24

print(f'Labels: {labels.min()} – {labels.max()}  |  {np.unique(labels).size} unique classes')

In [ ]:
# Re-grid labels back onto the original 3D cube coordinates
# valid_mask tracks which voxels had non-NaN features
label_volume = np.full(valid_mask.shape, np.nan)
label_volume[valid_mask] = labels

cube['SOM_classification'] = xr.DataArray(
    label_volume,
    dims=masked_data[FEATURE_COLS[0]].dims,
    coords=masked_data[FEATURE_COLS[0]].coords,
)

print('SOM classification added to cube.')

---
## Step 7 — Visualise Results

In [ ]:
# Cross-section through the well
plot_section(
    cube.SOM_classification,
    WELL_XLINE,
    slice(2000, 2350),
    'gist_ncar',
    cube,
    f'SOM Classification ({SOM_X}×{SOM_Y}) — xline {WELL_XLINE}',
)
plt.show()

In [ ]:
# Horizon map: interpolate SOM labels onto the Sandnes surface
sandnes_som = cube.SOM_classification.interp(
    {'samples': cube.Sandnes_TWT}, method='nearest'
)

fig, ax = plt.subplots(figsize=(12, 10))
mesh = ax.pcolormesh(
    sandnes_som.cdp_x.values,
    sandnes_som.cdp_y.values,
    sandnes_som.values,
    shading='auto',
    cmap='gist_ncar',
)
ax.plot(WELL_XY[0], WELL_XY[1], 'ro', markersize=8, label='Well')
ax.set_aspect('equal')
ax.legend()
ax.set_title('SOM Classification — Sandnes Horizon Map')
plt.colorbar(mesh, ax=ax, orientation='horizontal', label='SOM class')
plt.show()

In [ ]:
# SOM weight visualisation via PCA → RGB
weights = som.get_weights().reshape(-1, len(FEATURE_COLS))  # (SOM_X*SOM_Y, n_features)
pca_rgb = PCA(n_components=3).fit_transform(weights)

# Normalise each PC to [0, 1] for RGB display
pca_rgb -= pca_rgb.min(axis=0)
pca_rgb /= pca_rgb.max(axis=0)
rgb_map = pca_rgb.reshape(SOM_X, SOM_Y, 3)

fig, ax = plt.subplots(figsize=(5, 5))
ax.imshow(rgb_map, interpolation='nearest')
ax.set_title('SOM neurons coloured by PCA(weights) → RGB')
plt.show()

---
## Step 8 — Export Classification Cube to SEG-Y

In [ ]:
out_path = VOL_DIR / f'SOM_{SOM_X}x{SOM_Y}_BryneFm.segy'

cube.seisio.to_segy(
    str(out_path),
    data_var='SOM_classification',
    trace_header_map={'cdp_x': 73, 'cdp_y': 77},
    iline=189,
    xline=193,
)

print(f'Exported → {out_path}')